In [1]:
!pip install -q hvplot


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 15.7 MB/s eta 0:00:00


In [2]:
import pandas as  pd
import numpy as np
import matplotlib.pyplot as plt
from time import time
import hvplot.pandas

In [3]:
data = pd.read_csv('/content/data (1).csv')

In [4]:
data

,private,apps,accept,enroll,top10perc,top25perc,f_undergrad,p_undergrad,outstate,room_board,books,personal,phd,terminal,s_f_ratio,perc_alumni,expend,grad_rate
0,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,Yes,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,Yes,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772,No,2197,1515,543,4,26,3089,2029,6797,3900,500,1200,60,60,21.0,14,4469,40
773,Yes,1959,1805,695,24,47,2849,1107,11520,4960,600,1250,73,75,13.3,31,9189,83
774,Yes,2097,1915,695,34,61,2793,166,6900,4200,617,781,67,75,14.4,20,8323,49
775,Yes,10705,2453,1317,95,99,5217,83,19840,6510,630,2115,96,96,5.8,49,40386,99


In [5]:
data[data['grad_rate']>100]

,private,apps,accept,enroll,top10perc,top25perc,f_undergrad,p_undergrad,outstate,room_board,books,personal,phd,terminal,s_f_ratio,perc_alumni,expend,grad_rate
95,Yes,3847,3433,527,9,35,1010,12,9384,4840,600,500,22,47,14.3,20,7697,118


In [6]:
data.loc[data.grad_rate>100,'grad_rate']=100

In [7]:
data[data['grad_rate']>100]

,private,apps,accept,enroll,top10perc,top25perc,f_undergrad,p_undergrad,outstate,room_board,books,personal,phd,terminal,s_f_ratio,perc_alumni,expend,grad_rate


In [8]:
accuracy = {}
speed = {}

X = data.drop('private', axis=1)
y = data.private.map({"Yes": 1, "No": 0})

In [9]:
X.isnull().sum()

,0
apps,0
accept,0
enroll,0
top10perc,0
top25perc,0
f_undergrad,0
p_undergrad,0
outstate,0
room_board,0
books,0


In [10]:
X

,apps,accept,enroll,top10perc,top25perc,f_undergrad,p_undergrad,outstate,room_board,books,personal,phd,terminal,s_f_ratio,perc_alumni,expend,grad_rate
0,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772,2197,1515,543,4,26,3089,2029,6797,3900,500,1200,60,60,21.0,14,4469,40
773,1959,1805,695,24,47,2849,1107,11520,4960,600,1250,73,75,13.3,31,9189,83
774,2097,1915,695,34,61,2793,166,6900,4200,617,781,67,75,14.4,20,8323,49
775,10705,2453,1317,95,99,5217,83,19840,6510,630,2115,96,96,5.8,49,40386,99


In [11]:
data.private.value_counts()


,count
private,
Yes,565
No,212


# 1. GradientBoostingClassifier from Scikit-Learn
Gradient Boosting Classifier is a machine learning technique used for classification problems. It is an ensemble learning method that combines the predictions of multiple weak models to produce a strong overall prediction. The weak models are typically decision trees and the predictions are combined through a weighted average, where more weight is given to the trees that produce a better result on the training data. The weights are updated in each iteration of the boosting process, which adjusts the focus to the samples that are misclassified in the previous iteration. The final prediction is made by taking a weighted majority vote of all the trees in the ensemble.

In [12]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold


In [13]:
model = GradientBoostingClassifier()

In [14]:
start = time()
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
score = cross_val_score(model, X, y, scoring='f1', cv=cv, n_jobs=-1, error_score='raise')
speed['GradientBoosting'] = np.round(time() - start, 3)
accuracy['GradientBoosting'] = (np.mean(score) * 100).round(3)

In [15]:
print(f"Mean F1 score: {accuracy['GradientBoosting']}")
print(f"STD: {np.std(score):.3f}")
print(f"Run Time: {speed['GradientBoosting']}s")

Mean F1 score: 96.501
STD: 0.015
Run Time: 22.334s


# 2. XGBoost
XGBoost (eXtreme Gradient Boosting) is an optimized and scalable implementation of gradient boosting for tree-based models. It is designed for both efficiency and performance and is widely used for large-scale machine learning tasks such as classification and regression.

XGBoost is notable for its parallel processing capabilities, its ability to handle missing data, and its advanced features for model tuning. Additionally, it provides a flexible and expressive syntax for defining models, making it accessible to both novice and expert users.

In [16]:
from xgboost import XGBClassifier

model = XGBClassifier()

start = time()
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
score = cross_val_score(model, X, y, scoring='f1', cv=cv, n_jobs=-1)

speed['XGBoost'] = np.round(time() - start, 3)
accuracy['XGBoost'] = (np.mean(score) * 100).round(3)

print(f"Mean F1 score: {accuracy['XGBoost']}")
print(f"STD: {np.std(score):.3f}")
print(f"Run Time: {speed['XGBoost']}s")

Mean F1 score: 95.931
STD: 0.015
Run Time: 2.168s


# 3. LightGBM
LightGBM is a gradient boosting framework that uses tree-based learning algorithms. It is designed to be efficient and scalable for large-scale machine learning tasks, such as classification and regression. It is particularly well suited for working with large datasets, due to its fast training speed, low memory usage, and ability to handle missing values.

LightGBM uses a novel gradient-based one-side sampling technique to handle large datasets, which allows it to achieve high training speeds while still maintaining accuracy. It also includes advanced features for model tuning, such as support for parallel and GPU-based computing, as well as various objective functions for both binary and multiclass classification problems.

In [17]:
from lightgbm import LGBMClassifier

model = LGBMClassifier()

start = time()
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
score = cross_val_score(model, X, y, scoring='f1', cv=cv, n_jobs=-1)

speed['LGBM'] = np.round(time() - start, 3)
accuracy['LGBM'] = (np.mean(score) * 100).round(3)

print(f"Mean F1 score: {accuracy['LGBM']}")
print(f"STD: {np.std(score):.3f}")
print(f"Run Time: {speed['LGBM']}s")

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Mean F1 score: 95.916
STD: 0.015
Run Time: 10.552s


# 4. CatBoost
CatBoost is a gradient boosting library that is well suited for working with categorical data. It is an open-source library developed by Yandex and is designed to be fast and scalable for large datasets.

One of the main features of CatBoost is its ability to handle categorical features without the need for one-hot encoding. This is achieved by using an ordered encoding of the categorical features and incorporating the order information into the learning process. This approach results in more accurate models and reduced overfitting, compared to traditional one-hot encoding methods.

CatBoost also includes advanced features for model tuning, such as support for parallel computing, various loss functions, and an efficient implementation of the gradient boosting algorithm. It also provides a simple and intuitive API for defining and training models, making it accessible to both novice and expert users.

In [18]:
! pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.3 MB/s eta 0:00:00


In [19]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(silent=True)

start = time()
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
score = cross_val_score(model, X, y, scoring='f1', cv=cv, n_jobs=-1)

speed['CatBoost'] = np.round(time() - start, 3)
accuracy['CatBoost'] = (np.mean(score) * 100).round(3)


print(f"Mean F1 score: {accuracy['CatBoost']}")
print(f"STD: {np.std(score):.3f}")
print(f"Run Time: {speed['CatBoost']}s")

Mean F1 score: 96.181
STD: 0.010
Run Time: 97.461s


# 5. AdaBoost
AdaBoost (Adaptive Boosting) is a machine learning technique used for classification and regression problems. It is an ensemble learning method that combines the predictions of multiple weak models to produce a strong overall prediction. The weak models are typically decision trees with a shallow depth, and the predictions are combined through a weighted average, where more weight is given to the trees that produce a better result on the training data.

AdaBoost works by iteratively fitting weak models to the training data and adjusting the weights of the samples in each iteration to focus on the samples that are misclassified in the previous iteration. The final prediction is made by taking a weighted majority vote of all the trees in the ensemble.

In [20]:
from sklearn.ensemble import AdaBoostClassifier

model = AdaBoostClassifier()

start = time()
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
score = cross_val_score(model, X, y, scoring='f1', cv=cv, n_jobs=-1)

speed['AdaBoost'] = np.round(time() - start, 3)
accuracy['AdaBoost'] = (np.mean(score) * 100).round(3)

print(f"Mean F1 score: {accuracy['AdaBoost']}")
print(f"STD: {np.std(score):.3f}")
print(f"Run Time: {speed['AdaBoost']}s")

Mean F1 score: 95.541
STD: 0.015
Run Time: 7.698s


# 6. Scikit-Learn vs XGBoost vs LightGBM vs CatBoost

In [21]:
for algo, result in accuracy.items():
    print(f"{algo:{20}}: Score: {result}, Speed: {speed[algo]}")

GradientBoosting    : Score: 96.501, Speed: 22.334
XGBoost             : Score: 95.931, Speed: 2.168
LGBM                : Score: 95.916, Speed: 10.552
CatBoost            : Score: 96.181, Speed: 97.461
AdaBoost            : Score: 95.541, Speed: 7.698


In [22]:
accuracy_df = pd.DataFrame(list(accuracy.items()), columns=['Algorithm', 'Accuracy'])

speed_df = pd.DataFrame(list(speed.items()), columns=['Algorithm', 'Time'])

In [23]:
accuracy_df.hvplot.barh(x='Algorithm', y='Accuracy')


:Bars   [Algorithm]   (Accuracy)

In [24]:
speed_df.hvplot.barh(x='Algorithm', y='Time')


:Bars   [Algorithm]   (Time)